In [ ]:
print('========= THE ANALYSIS TABLE - standard  DUCKDB ')

In [ ]:
# IMPORTS  single parquet file  TO DUCKDB
import pandas as pd
import numpy as np
import duckdb
from pathlib import Path
from binance.client import Client
from datetime import datetime
import time
import plotly.graph_objects as go 
import plotly.express as px

# ------------- configure dataframe printing layout 
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)
#------------------------------------------------

# single file path

# pqt_file = Path.cwd()/'__inputs'/'sol_monthly_aggts_2026'/'SOLUSDT-aggTrades-2026-01.parquet'
pqt_file = Path.cwd()/'__inputs'/'BTCUSDT-aggTrades-2026-02.parquet'
# symbol name
SYMBOL = pqt_file.name.split("-")[0]

# chart naming
fname = datetime.now().strftime("%Y%m%d_%H%M%S")
stt =datetime.now() # start time

print(SYMBOL)
# print(pqt_file)
print(pqt_file.exists())


In [ ]:
# coin stats like average volume from BINANCE

client = Client()
klines = client.futures_klines(symbol=SYMBOL, interval='1d', limit=21)
closed_klines = klines[:-1]  # Exclude today's incomplete candle
# Index 4: Close Price, Index 5: Volume
volumes = [float(k[5]) for k in closed_klines]
prices = [float(k[4]) for k in closed_klines]
# ---------------------------------
vol_20_ma = sum(volumes[-20:]) / 20
price_20_ma = sum(prices[-20:]) / 20
notional_ma = vol_20_ma * price_20_ma

vol = (f"\n volume ma for {SYMBOL} : $ {notional_ma:,.2f} \n")

In [ ]:
# LOAD  aggregated DATA TO dataframe with  DUCKDB

conn = duckdb.connect(database=":memory:")

qry = f"""--sql
SELECT
    time_bucket(INTERVAL '5 Minutes', epoch_ms(CAST(transact_time AS BIGINT))) AS transact_time,
    arg_min(price, transact_time) AS tde1_price,
    arg_max(price, transact_time) AS last_price,
    sum(CASE WHEN is_buyer_maker = FALSE THEN (price * quantity) ELSE 0.0 END) AS buyVol_usdt,
    sum(CASE WHEN is_buyer_maker = TRUE  THEN (price * quantity) ELSE 0.0 END) AS sellVol_usdt
FROM read_parquet('{pqt_file}')
GROUP BY 1
ORDER BY 1 ASC;
"""

bdf = conn.execute(qry).df()
conn.close()

print(bdf.head())

In [ ]:
# the Full PANDAS DATAFRAME

# rename a column
bdf = bdf.rename(columns={'transact_time': 'time'})
# print(bdf.head())

# derived columns  ------------------------------------- PERCENTAGES
bdf['delta'] = (bdf['buyVol_usdt'] - bdf['sellVol_usdt']).round(3)
bdf['prc_change_window'] = ((
    bdf['last_price']-bdf['tde1_price'])/bdf['tde1_price'])*100
day_open_price = bdf['tde1_price'].iloc[0]
bdf['dayOpen_prcchange'] = ((
    bdf['last_price'] - day_open_price)/day_open_price)*100

# core derived columns
bdf['s_delta'] = (bdf['delta']/notional_ma)*100
bdf['cumulative_sd'] = bdf['s_delta'].cumsum()

bdf['mkt_eff'] = (np.where(bdf['s_delta'] != 0,
                    (bdf['dayOpen_prcchange']/bdf['s_delta']), 0.00))
bdf['mkt_eff'] = (bdf['mkt_eff']*100).abs()

bdf['mkt_regime'] = (np.where(bdf['cumulative_sd'] != 0,
                        (bdf['dayOpen_prcchange']/bdf['cumulative_sd']), 0.00))
bdf['mkt_regime'] = (bdf['mkt_regime']*100).abs()

# ----------COLUMN FORMATING ------------------------
bdf['delta'] = bdf['delta'] #.round(2)
bdf['buyVol_usdt'] = bdf['buyVol_usdt']#.round(2)
bdf['sellVol_usdt'] = bdf['sellVol_usdt']#.round(2)

# bdf.to_parquet(file_output_path)

print(bdf.head())

In [ ]:
 # THE TREND GRAPH

fig1= go.Figure()
fig1.add_trace(go.Scatter(x=bdf.time, y=bdf.dayOpen_prcchange, mode='lines', line=dict(color='purple'), name='change 4rm day_opn' ))

In [ ]:
# time taken overall
# time_taken =datetime.now()- stt
# print(time_taken)

In [ ]:
#- -----------  MARKET PRESSURE GRAPH --------------

# fig2 =px.line(bdf, x='time', y='s_delta', title='MARKET PRESSURE')

fig2= go.Figure()
fig2.add_trace(go.Scatter(x=bdf.time, y=bdf.s_delta, mode='lines', line=dict(color='blue'), name='% delta' ))
fig2.add_trace(go.Scatter(x=bdf.time, y=bdf.prc_change_window, mode='lines', line=dict(color="#dd4508"), name='5m prc change' ))
# fig2.update_layout(title= 'MARKET PRESSURE GRAPH' )

In [ ]:
#- -----------  MARKET PRESSURE FLOW GRAPH --------------
# fig3= px.line(bdf, x='time', y='cumulative_sd', title='MARKET PRESSURE FLOW')
fig3= go.Figure()
fig3.add_trace(go.Scatter(x=bdf.time, y=bdf.cumulative_sd, mode='lines+markers', line=dict(color='green'), name='cumulative pctg delta'))
# fig3.update_layout(title= 'MARKET PRESSURE FLOW GRAPH' )

In [ ]:
# --------------------- OUTLIER GRAPHS
# -------------- MARKET EFFICIENCY GRAPH -----------------
# fig4= px.line(bdf, x='time', y='mkt_eff', title='MARKET efficiency GRAPH')
fig4= go.Figure()
fig4.add_trace(go.Scatter(x=bdf.time, y=bdf.mkt_eff, mode='lines', line=dict(color='black'), name='mkt eff'))

# -------------- MARKET price  GRAPH -----------------
# fig6= px.line(bdf, x='time', y='tde1_price', title='price')
fig6= go.Figure()
fig6.add_trace(go.Scatter(x=bdf.time, y=bdf.tde1_price, mode='lines', line=dict(), name='first price'))
# fig6.show()

# -------------- MARKET REGIME STRUCTURE GRAPH -----------------
# fig5= px.line(bdf, x='time', y='mkt_regime', title='MARKET structure')
fig5= go.Figure()
fig5.add_trace(go.Scatter(x=bdf.time, y=bdf.mkt_regime, mode='lines', line=dict(color='black'), name='mkt regime'))